In [ ]:
import os; os.chdir(os.path.dirname(os.getcwd())) if os.path.basename(os.getcwd()) == "notebooks" else None

# 03 - First experiment: does feedback matter?

**Hypothesis.** A feedback policy (PI controller, or an RL policy conditioned on the
plasma state) should degrade more gracefully than an open-loop schedule when the plant
deviates from the nominal ITER-hybrid TORAX config, because it can react to the
realised state instead of replaying a fixed time trajectory. The PI controller only
feeds back one scalar (central current density); an RL policy that sees the full
`"profiles"` observation could in principle do better, but may also not learn to use it.

**Protocol.** Perturb the initial conditions or the transport model (e.g. a random
multiplier on `transport.chi_min`/`chi_max`, or on the initial `T_e`/`T_i`/`n_e` levels)
before each episode's TORAX simulator is built. Evaluate `PIController`, the best
open-loop schedule, and a trained RL policy on the *same* sequence of perturbation seeds
(matched pairs, not independently resampled), and compare benchmark return and failure
rate as the perturbation strength grows.

## Perturbed-environment scaffold

Gym-TORAX bakes its simulator's config into the environment at construction time:
`BaseEnv.__init__` calls `_get_torax_config()` once and builds TORAX from its `"config"`
dict (see `gymtorax.envs.base_env.BaseEnv._get_torax_config`,
`gymtorax.envs.iter_hybrid_env.IterHybridEnv._get_torax_config`, and the raw config at
`gymtorax.envs.iter_hybrid_env.CONFIG`). So a per-episode perturbation cannot be applied
after the fact via `RampupEnv.reset()` alone - it requires a fresh `IterHybridEnv`
subclass instance per episode, with `_get_torax_config()` returning a perturbed config.

In [ ]:
import copy

import numpy as np
from gymtorax import IterHybridEnv
from gymtorax.envs.iter_hybrid_env import CONFIG

from rl_tokamak.env import EnvConfig, RampupEnv


def perturbed_config(rng: np.random.Generator, scale: float) -> dict:
    """Deep copy of gymtorax's ITER-hybrid CONFIG with one parameter randomly scaled.

    TODO: verify these key names against the installed gymtorax 1.0.0 source
    (gymtorax.envs.iter_hybrid_env.CONFIG) - the config dict layout is not part of
    gymtorax's public API and could change between versions.
    """
    cfg = copy.deepcopy(CONFIG)
    factor = 1.0 + rng.uniform(-scale, scale)
    # TODO: pick ONE perturbation axis per experiment (mixing several confounds the result).
    # Transport strength (turbulent heat/particle diffusivity floor):
    cfg["transport"]["chi_min"] *= factor
    # Or initial-condition levels instead (uncomment, and remove the transport line above):
    # cfg["profile_conditions"]["T_e"][0.0][0.0] *= factor
    # cfg["profile_conditions"]["T_i"][0.0][0.0] *= factor
    # cfg["profile_conditions"]["n_e"][0][0.0] *= factor
    return cfg


class _RandomizedIterHybridEnv(IterHybridEnv):
    """IterHybridEnv whose TORAX config is perturbed once, at construction time."""

    def __init__(self, rng: np.random.Generator, scale: float, **kwargs):
        self._rng, self._scale = rng, scale
        super().__init__(**kwargs)

    def _get_torax_config(self):
        return {"config": perturbed_config(self._rng, self._scale), "discretization": "fixed", "ratio_a_sim": 1}


class PerturbedRampupEnv(RampupEnv):
    """RampupEnv whose inner TORAX simulator is rebuilt, freshly perturbed, per episode.

    TODO: this is a scaffold, not a finished implementation:
    - `_get_torax_config()` is only read once, inside `IterHybridEnv.__init__` (see the
      note above), so `reset()` below still replays the SAME perturbation drawn in
      `__init__` - rebuild `self.inner` with a new `_RandomizedIterHybridEnv` in `reset()`
      to actually draw a fresh perturbation every episode.
    - rebuilding a TORAX simulator costs ~10 s (the environment "build" cost noted
      elsewhere in this project); confirm that is acceptable for the planned number of
      perturbation seeds x policies before running a real sweep.
    """

    def __init__(self, config: EnvConfig | None = None, scale: float = 0.2, seed: int = 0):
        self.scale = scale
        self.rng = np.random.default_rng(seed)
        inner = _RandomizedIterHybridEnv(self.rng, self.scale, render_mode=None, log_level="critical")
        super().__init__(config, inner=inner)

    def reset(self, *, seed=None, options=None):
        # TODO: rebuild self.inner with a fresh _RandomizedIterHybridEnv here before calling
        # super().reset(), so every episode gets a newly sampled perturbation.
        return super().reset(seed=seed, options=options)

## Evaluation loop

Scores `PIController`, `OpenLoopController` and a loaded RL policy over `N_SEEDS`
perturbation draws at a fixed `PERTURBATION_SCALE`. `PerturbedRampupEnv` is not finished
(see TODOs above), so this loop currently records `None` returns rather than building a
perturbed environment - fill in the TODOs below once it is.

In [ ]:
import pandas as pd

from rl_tokamak.controllers import OpenLoopController, PIController
from rl_tokamak.evaluate import run_controller, run_policy
from rl_tokamak.policies import load_policy

N_SEEDS = 10
PERTURBATION_SCALE = 0.2  # TODO: sweep this, e.g. [0.0, 0.1, 0.2, 0.3], once perturbations work
RL_RUN_DIR = None  # TODO: point this at a trained checkpoint, e.g. Path("data/runs/sac_s0")

rows = []
for seed in range(N_SEEDS):
    # TODO: replace with a working perturbed environment, e.g.
    # env = PerturbedRampupEnv(EnvConfig(), scale=PERTURBATION_SCALE, seed=seed)
    # once PerturbedRampupEnv actually perturbs the TORAX config per episode (see TODOs above).
    env = None

    for policy_name in ("pi", "open_loop", "rl"):
        if env is None:
            rows.append({"seed": seed, "policy": policy_name, "scale": PERTURBATION_SCALE,
                        "benchmark_return": None, "failed": None})
            continue
        if policy_name == "pi":
            result = run_controller(env, PIController())
        elif policy_name == "open_loop":
            result = run_controller(env, OpenLoopController())
        else:
            if RL_RUN_DIR is None:
                continue
            _, policy_fn = load_policy(RL_RUN_DIR)
            result = run_policy(env, policy_fn)
        rows.append({"seed": seed, "policy": policy_name, "scale": PERTURBATION_SCALE,
                    "benchmark_return": result["benchmark_return"], "failed": result["failed"]})
        env.close()

perturbation_df = pd.DataFrame(rows)
perturbation_df

## Plot

Mean benchmark return per policy at the current `PERTURBATION_SCALE`.

In [ ]:
import matplotlib.pyplot as plt

# TODO: once PERTURBATION_SCALE is swept, plot mean return (and failure rate) vs scale per
# policy instead of this single-scale bar chart.
fig, ax = plt.subplots(figsize=(6, 4))
scored = perturbation_df.dropna(subset=["benchmark_return"])
if len(scored):
    stats = scored.groupby("policy")["benchmark_return"].agg(["mean", "std"])
    ax.bar(stats.index, stats["mean"], yerr=stats["std"].fillna(0.0))
    ax.set_ylabel(f"benchmark return (perturbation scale = {PERTURBATION_SCALE})")
else:
    ax.text(0.5, 0.5, "no perturbation results yet - finish the TODOs above", ha="center", va="center",
            transform=ax.transAxes)
fig.tight_layout()
fig

## What would count as "feedback matters"

Plot mean benchmark return and failure rate against perturbation scale for all three
policies. Feedback matters if the open-loop schedule's return falls off fastest (it
cannot react to the realised initial state or transport at all), and the RL policy's
return stays flat for longer than the PI controller's, or fails less often at the same
scale - meaning conditioning on the full state buys more robustness than the PI
controller's single fed-back scalar (central current density). If the RL curve degrades
as fast as open-loop, or PI matches RL at every scale, this benchmark is not rewarding
feedback beyond a single hand-picked sensor.